# Session 4 · Your first mini-agent

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)

In [ ]:
from course_llm import tool
import json

# 1. THE TOOL: plain Python. Lumo track: the sandbox order system. Own case: your own data here.
ORDERS = {o["order_id"]: o for o in lumo.orders()}

def lookup_order(order_id):
    o = ORDERS.get(str(order_id).strip("# "))
    return json.dumps({k: o[k] for k in ("status", "carrier", "shipped_on", "delivered_on")}) if o else "No order found."

# 2. THE TOOL DESCRIPTION: all the model knows about the tool. Words matter here.
lookup = tool("lookup_order", "Look up the status of a customer order by its order number.",
              {"order_id": ("string", "The order number, e.g. 1047")}, lookup_order)

# 3. THE INSTRUCTIONS
system = "You are Lumo's support assistant. Check the order system before you answer questions about an order. Be brief and friendly."

# 4. THE LOOP: llm.run_agent asks the model, runs any tool it asks for, sends the result back, repeats.
print(llm.run_agent("Hi, where is my order #1047?", tools=[lookup], system=system, max_steps=5))

In [ ]:
# Watch the agent DECIDE: no tool needed for the first question; two lookups (one missing) for the second
print(llm.run_agent("What are your opening hours?", tools=[lookup], system=system))
print()
print(llm.run_agent("Can you check orders 1061 and 9999 for me?", tools=[lookup], system=system))